# FIAP • ENGENHARIA DA COMPUTAÇÃO (Turma 1ECR • 2022)
## Disciplina: Indústria 4.0
### Checkpoint 4: Automação e Controle de Motores de Passo com Microcontrolador

---

### 1. Enunciado do Problema e Código Fonte Oficial
- **Hardware:** Microcontrolador Arduino / ESP32 acoplado a motor de passo unipolar 28BYJ-48 e driver com circuito integrado Darlington ULN2003.
- **Arquivo de Código Fonte Oficial:** `Industria 4.0 (1).ino`
- **Objetivo Técnico:** Projetar o acionamento nas modalidades de Passo Completo (Full-Step de 4 fases) e Meio-Passo (Half-Step de 8 fases), garantindo torque de retenção e posicionamento milimétrico em células de triagem automática e fresadoras CNC.

---

### 2. Resolução Matemática Passo a Passo (Padrão Symbolab)
#### Cinemática de Posicionamento e Resolução Angular

**Passo 1: Dedução da Resolução Angular do Motor com Redução**
$$\theta_{\text{passo}} = \frac{360^\circ}{N_{\text{passos,int}} \cdot R_{\text{red}}}$$
$$\text{Onde: } N_{\text{passos,int}} = \text{passos internos por volta do rotor}, \quad R_{\text{red}} = \text{razão da caixa de redução}$$

**Passo 2: Parâmetros Construtivos do Motor 28BYJ-48**
- Ângulo de Passo do Rotor Interno: $\alpha = 5{,}625^\circ$ no modo Meio-Passo
- Passos Internos por Volta: $N_{\text{half}} = \frac{360^\circ}{5{,}625^\circ \times 0{,}5} = 64 \times 2 = 128\text{ passos}$
- Relação Exata da Caixa de Redução: $R_{\text{red}} = 64$
- Total de Passos no Eixo de Saída por Volta Completa:
  $$N_{\text{total}} = 128 \times 64 = 4.096\text{ passos/volta}$$

**Passo 3: Substituição Numérica da Resolução Angular**
$$\theta_{\text{resolucao}} = \frac{360^\circ}{4.096} = 0{,}08789^\circ \text{ por meio-passo}$$

**Passo 4: Cálculo da Frequência de Pulsos para Velocidade Alvo**
Para uma velocidade angular desejada de $\omega = 15\text{ RPM}$ no eixo de saída:
$$f_{\text{pulsos}} = \frac{\text{RPM} \cdot N_{\text{total}}}{60} = \frac{15 \cdot 4.096}{60} = \frac{61.440}{60} = 1.024\text{ Hz}$$
$$T_{\text{delay}} = \frac{1}{f_{\text{pulsos}}} = \frac{1}{1.024\text{ s}} \approx 976{,}56\,\mu\text{s} \approx 1\text{ ms por passo}$$

**Passo 5: Resultado Final e Parametrização de Firmware**
$$\mathbf{\theta_{\text{res}} = 0{,}0879^\circ/\text{passo} \quad | \quad f = 1.024\text{ Hz} \quad (\text{Delay entre pulsos: } 976\,\mu\text{s})}$$
> **Aplicação em Firmware:** O temporizador do microcontrolador deve disparar as comutações das 4 bobinas no vetor `B1000, B1100, B0100, B0110, B0010, B0011, B0001, B1001` a cada 1 ms para garantir movimentação contínua sem perda de passo por ressonância.


In [ ]:
# Cálculo Cinemático de Motor de Passo
passos_por_volta = 4096 # Meio-passo com redução 1:64
res_angular_graus = 360.0 / passos_por_volta

rpm_desejado = 15.0
freq_pulsos_hz = (rpm_desejado * passos_por_volta) / 60.0
delay_microsegundos = (1.0 / freq_pulsos_hz) * 1e6

print(f"Resolução Angular no Eixo: {res_angular_graus:.5f}° / passo")
print(f"Frequência de Comutação:     {freq_pulsos_hz:.2f} Hz")
print(f"Delay entre Passos:         {delay_microsegundos:.1f} microsegundos")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rpms = np.linspace(1, 25, 100)
freqs = (rpms * passos_por_volta) / 60.0

plt.figure(figsize=(8.5, 4.5))
plt.plot(rpms, freqs, color='#ff7b25', linewidth=2.5, label='Frequência de Pulso (Hz)')
plt.axvline(15.0, color='#00f0ff', linestyle='--', label='Ponto de Operação Nominal (15 RPM)')
plt.xlabel('Velocidade no Eixo de Saída (RPM)', fontsize=11)
plt.ylabel('Frequência de Comutação do Driver (Hz)', fontsize=11)
plt.title('FIAP 1ECR • Curva de Frequência do Motor 28BYJ-48 (CP4)', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('grafico_resolucao.png', dpi=150)
plt.show()
